# Case pipeline_lab — Churn (Databricks)

Funil completo, uma etapa por célula: `divisao → construcao → esfera1 → esfera2 → categorizar → preselecao → treinamento`.

Cada chamada mostra **todos os parâmetros**, mesmo os que ficam no valor default — assim serve de referência do que dá pra ajustar. Ver `python/pipeline_lab/REFERENCIA.md` (ou o PDF em `docs/pipeline_lab_referencia.pdf`) para a explicação de cada parâmetro e a literatura por trás.

## 0. Instalação

In [ ]:
%pip install --upgrade --force-reinstall git+https://github.com/PedroMaiorano/modelagem-lab.git

In [ ]:
dbutils.library.restartPython()

## 1. Carregar os dados (Spark → pandas)

`sample_submission`/`test` são a base sem rótulo (Kaggle-style) — usadas só no fim, para gerar a submissão. O treinamento roda inteiro sobre `train`, que tem `Churn`.

In [ ]:
sample_submission = spark.table("workspace.default.churn_sample_submission")
train = spark.table("workspace.default.churn_train")
test = spark.table("workspace.default.churn_test")

## 2. Preparar o alvo e o split dev/teste

`Churn` ("Yes"/"No") vira 0/1. O split usa uma única coluna de sorteio (`_sorteio`) reaproveitada nas duas comparações — chamar `F.rand(seed=42)` de novo em cada `.when()` sorteia números **diferentes** a cada chamada, mesmo com a mesma seed, e quebra a proporção 70/15/15 pretendida (o segundo corte vira independente do primeiro, não condicional).

In [ ]:
from pyspark.sql import functions as F

train = train.withColumn("Churn", F.when(F.col("Churn") == "Yes", 1).otherwise(0))

train = train.withColumn("_sorteio", F.rand(seed=42))
train = train.withColumn(
    "split",
    F.when(F.col("_sorteio") < 0.70, "train")
     .when(F.col("_sorteio") < 0.85, "val")
     .otherwise("test"),
).drop("_sorteio")

train_pd = train.toPandas()
test_pd = test.toPandas()

train_pd["split"].value_counts()

## 3. Divisão dev/teste (`pipeline_lab.divisao`)

`dividir_por_amostra` usa a coluna `split` que já veio do passo anterior. `coluna_y="Churn"` já renomeia o alvo para `"y"` — nome que todo o resto do funil exige. O bucket `"val"` fica de fora deste split (reservado como holdout extra — ver nota no fim do notebook, a lib ainda não expõe uma função de "aplicar transformação em dado novo sem re-ajustar" para um terceiro conjunto).

In [ ]:
from pipeline_lab import categorizar, construcao, divisao, esfera2, preselecao, treinamento

df_dev, df_teste = divisao.dividir_por_amostra(
    df=train_pd,
    coluna_amostra="split",
    valores_dev=["train"],
    valores_teste=["test"],
    coluna_y="Churn",
)

print(f"dev: {len(df_dev)} linhas | teste: {len(df_teste)} linhas")

## 4. Construção de variáveis (`pipeline_lab.construcao`) — opcional

Razões/diferenças interpretáveis entre pares de colunas relacionadas (ex.: `TotalCharges / tenure` ≡ "gasto médio mensal"). **Ajuste `pares` para os nomes reais das suas colunas** — deixei vazio por padrão (não quebra nada, só não adiciona candidata nenhuma) porque não sei o schema exato de `churn_train` neste ambiente.

In [ ]:
# Exemplo (ajuste os nomes de coluna pro seu dataset e descomente):
# pares = [("TotalCharges", "tenure", "gasto_medio_mensal")]
pares: list[tuple[str, str, str]] = []

if pares:
    novas_dev = construcao.construir_razoes_em_lote(df=df_dev, pares=pares, epsilon=1e-6)
    novas_teste = construcao.construir_razoes_em_lote(df=df_teste, pares=pares, epsilon=1e-6)
    df_dev = pd.concat([df_dev, novas_dev], axis=1)
    df_teste = pd.concat([df_teste, novas_teste], axis=1)

df_dev.shape, df_teste.shape

## 5. Esfera 1 — agregação temporal (`pipeline_lab.esfera1`) — não aplicável aqui

Esfera 1 exige um **painel** (várias linhas por chave ao longo do tempo — ex.: atraso mês a mês do mesmo cliente). O dataset de churn aqui é uma linha por cliente (snapshot), não um painel, então esta etapa é pulada. Fica só como referência de assinatura, caso um dia você tenha um painel de uso mensal do cliente para agregar antes do resto do funil:

```python
from pipeline_lab import esfera1

df_dev, df_teste, colunas_geradas = esfera1.aplicar(
    df_dev=df_dev,
    df_teste=df_teste,
    chave="id_cliente",
    coluna_tempo="safra",
    colunas_valor=["uso_mensal"],
    janelas=[3, 6, 12],
)
```

## 6. Esfera 2 — descoberta de interação (`pipeline_lab.esfera2`)

RuleFit-style: descobre regras tipo "A > x E B > y" via um ensemble de árvores rasas, materializa como coluna 0/1.

In [ ]:
df_dev, df_teste, colunas_regra = esfera2.aplicar(
    df_dev=df_dev,
    df_teste=df_teste,
    colunas_categoricas=None,
    profundidade_maxima=2,
    n_arvores=60,
    min_suporte=0.02,
    max_suporte=0.5,
    max_regras=10,
    permitir_cruzamento_entre_bases=True,
    proporcao_variaveis_por_split=None,
    iv_minimo=0.02,
)

print(f"Regras de interação estáveis encontradas: {len(colunas_regra)}")
colunas_regra

## 7. Categorização + WOE (`pipeline_lab.categorizar`)

Sempre a última etapa antes da pré-seleção/treinamento — gera as versões `_woe`/`_log`/`_bin`/etc. de cada candidata.

In [ ]:
def _log_progresso(coluna: str, iv: float) -> None:
    print(f"  {coluna}: IV={iv:.4f}")


woe_dev, woe_teste, iv_por_variavel = categorizar.categorizar_e_transformar(
    df_dev=df_dev,
    df_teste=df_teste,
    gerar_transformacoes_potencia=True,
    gerar_bin_ordinal=True,
    ao_processar_coluna=_log_progresso,
)

woe_dev.shape, woe_teste.shape

## 8. Pré-seleção (`pipeline_lab.preselecao`)

Filtra variância → IV → correlação, nesta ordem, antes de entregar ao Pedro_Wise.

In [ ]:
resultado_selecao = preselecao.pre_selecionar(
    df=woe_dev,
    iv_por_base=iv_por_variavel,
    limiar_variancia=1e-6,
    limiar_iv=0.02,
    limiar_correlacao=0.9,
)

print(
    f"candidatas: {resultado_selecao['n_inicial']} -> "
    f"{resultado_selecao['n_apos_variancia']} (variância) -> "
    f"{resultado_selecao['n_apos_iv']} (IV) -> "
    f"{resultado_selecao['n_final']} (correlação)"
)

woe_dev = woe_dev[[*resultado_selecao["colunas_mantidas"], "y"]]
woe_teste = woe_teste[[*resultado_selecao["colunas_mantidas"], "y"]]

## 9. Treinamento — Pedro_Wise (`pipeline_lab.treinamento`)

A busca stepwise de verdade (forward/backward, níveis 1 a 3). `criterio="teste"` é o default recomendado (evita o viés de aceitar ruído que `criterio="dev"` tem — ver `REFERENCIA.md`).

In [ ]:
resultado = treinamento.treinar(
    df_dev=woe_dev,
    df_teste=woe_teste,
    criterio="teste",
    shadow_probing=False,
    forward_simples=True,
    transformacao_simples_nivel1=True,
    backward_simples_nivel1=True,
    min_vars_para_backward=5,
    forward_duplo=True,
    forward_triplo=True,
    transformacao_simples_nivel2=True,
    backward_simples_nivel2=True,
    n_best_duplo=5,
    n_best_triplo_1=3,
    n_best_triplo_2=3,
    nivel3_ativado=False,
    n_best_backward=2,
    profundidade_maxima_nivel3=2,
    p_valor_maximo=None,
)

print(f"Variáveis selecionadas: {resultado.variaveis}")
print(f"KS dev:   {resultado.ks_dev:.4f}")
print(f"KS teste: {resultado.ks_teste:.4f}")
print(f"AUC teste: {resultado.auc_teste:.4f}")
print(f"Taxa de evento dev/teste: {resultado.taxa_evento_dev:.2%} / {resultado.taxa_evento_teste:.2%}")

## 10. Resultados

In [ ]:
import pandas as pd

print("Coeficientes:")
display(pd.Series(resultado.coeficientes, name="coeficiente").to_frame())

print("Estatísticas (coeficiente / erro padrão / p-valor — diagnóstico, não influencia a seleção):")
display(pd.DataFrame(resultado.estatisticas).T)

print("Tabela de decis (gains/KS table) na base de teste:")
display(pd.DataFrame(resultado.tabela_decis))

## Notas e próximos passos

- **Bucket `"val"`**: reservado de propósito, não usado neste notebook. Hoje `pipeline_lab.categorizar` ajusta e aplica a transformação WOE numa única chamada (`categorizar_e_transformar`) — não existe ainda uma função separada de "reaplicar as tabelas já ajustadas em dado novo, sem reajustar". Rodar `categorizar_e_transformar(val, val)` recalcularia o WOE usando o `y` do próprio `val` (vazamento). Se quiser avaliar nesse terceiro holdout ou gerar a submissão em cima de `churn_test` (sem rótulo), é preciso expor esse `aplicar_transformacoes(novos_dados, tabelas_do_dev)` na biblioteca antes — posso construir isso a seguir se for útil.
- **`test_pd`** (carregado no passo 1, vindo de `workspace.default.churn_test`) não foi usado neste notebook pelo mesmo motivo acima.
- Para ajustar qualquer hiperparâmetro, edite o valor no parâmetro nomeado correspondente em cada célula — todos já estão explícitos, não há nada escondido em default silencioso.